In [28]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from  langchain_core.messages import HumanMessage,BaseMessage
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [29]:
load_dotenv()

True

In [30]:
model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [31]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage],add_messages]

In [32]:
def chat_node(state:ChatState):

    messages=state['messages']

    response=model.invoke(messages)

    return {'messages':[response]}

In [33]:
checkpointer=MemorySaver()

graph= StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [36]:
initial_state={
    'messages':[HumanMessage(content='What is the capital of india')]
}

final_state=chatbot.invoke(initial_state)

final_state['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [37]:
thread_id = '1'

while True:
    user_message=input('Typed here: ')

    print('User: ',user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break

    config = {'configurable':{'thread_id':thread_id}}

    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    
    print("AI: ",response['messages'][-1].content)

User:  Hi my name is sumanta
AI:  Hello Sumanta! It's nice to meet you. How can I help you today?
User:  What is my name 
AI:  Your name is Sumanta.
User:  can you sort 1, 12,123, 1234, 13 ,11 ,101,16,112
AI:  Okay, here are those numbers sorted in ascending order:

1, 11, 12, 13, 16, 101, 112, 123, 1234
User:  how you sort this number
AI:  I sorted them **numerically**, from the smallest value to the largest value (this is called **ascending order**).

Here's how it works:

1.  **Compare Values:** I look at all the numbers and compare their actual numerical value.
2.  **Find the Smallest:** I identify the number with the lowest value.
3.  **Place It First:** That number goes at the beginning of the sorted list.
4.  **Repeat:** I then look at the *remaining* numbers, find the smallest among them, and place it next in the list.
5.  **Continue** this process until all numbers have been placed in order.

For your list: 1, 12, 123, 1234, 13, 11, 101, 16, 112

*   **1** is the smallest.
*  

In [38]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi my name is sumanta', additional_kwargs={}, response_metadata={}, id='54437117-1cfe-49ac-8837-40c74850e722'), AIMessage(content="Hello Sumanta! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c821b-5079-75a1-8b9c-4ce3be8e5ec4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 439, 'total_tokens': 446, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 420}}), HumanMessage(content='What is my name ', additional_kwargs={}, response_metadata={}, id='54cb2673-6c1a-4722-8a5e-d777a131ba58'), AIMessage(content='Your name is Sumanta.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id=